# Kiểm thử 2 — Robustness

Đánh giá độ bền của mô hình khi ảnh đầu vào bị suy giảm chất lượng:
JPEG compression, resize (thu nhỏ rồi phóng lại), Gaussian blur.

> **Ghi chú:** Đây là notebook **tham khảo** để chạy lại trên máy local. Kết quả số chính thức trong báo cáo (`reports/robustness/`) được tạo từ bản chạy trên Kaggle (`04-robustness-test-kaggle.ipynb`) với GPU. Notebook này tính đầy đủ Accuracy / Precision / Recall / F1 / AUC (giống bản Kaggle) và có thanh tiến trình `tqdm`.

In [ ]:
# Cell 1 — Import & Config
import os, sys, io, json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image, ImageFilter
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print(f"torch: {torch.__version__}")
print(f"timm: {timm.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Đảm bảo CWD là project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"CWD: {os.getcwd()}")

CHECKPOINT_PATH = "artifacts/checkpoints/best_stage3.pth"
TEST_CSV        = "data/splits/test.csv"
BATCH_SIZE      = 64
OUTPUT_DIR      = "reports/robustness"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_WORKERS = 4 if torch.cuda.is_available() else 0
PIN_MEMORY  = torch.cuda.is_available()

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

In [ ]:
# Cell 2 — Perturbation functions
def apply_jpeg_compression(img_pil, quality):
    buf = io.BytesIO()
    img_pil.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def apply_resize_downup(img_pil, scale):
    w, h    = img_pil.size
    small_w = max(1, int(w * scale))
    small_h = max(1, int(h * scale))
    small   = img_pil.resize((small_w, small_h), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)


def apply_gaussian_blur(img_pil, sigma):
    return img_pil.filter(ImageFilter.GaussianBlur(radius=sigma))


PERTURBATIONS = {
    "baseline":  lambda img: img,
    "jpeg_q70":  lambda img: apply_jpeg_compression(img, 70),
    "jpeg_q50":  lambda img: apply_jpeg_compression(img, 50),
    "jpeg_q30":  lambda img: apply_jpeg_compression(img, 30),
    "resize_50": lambda img: apply_resize_downup(img, 0.50),
    "resize_25": lambda img: apply_resize_downup(img, 0.25),
    "blur_s1":   lambda img: apply_gaussian_blur(img, 1.0),
    "blur_s2":   lambda img: apply_gaussian_blur(img, 2.0),
}

print("Perturbations:", list(PERTURBATIONS.keys()))


In [ ]:
# Cell 2b — Visualization ảnh mẫu trước/sau perturbation
# Lấy 1 ảnh real và 1 ảnh fake từ test set để minh họa
_df_preview = pd.read_csv(TEST_CSV)
_real_path = _df_preview[_df_preview["label"] == "Real"]["image_path"].iloc[0]
_fake_path = _df_preview[_df_preview["label"] == "Fake"]["image_path"].iloc[0]

_perturb_order = ["baseline", "jpeg_q70", "jpeg_q50", "jpeg_q30",
                  "resize_50", "resize_25", "blur_s1", "blur_s2"]
_labels = {
    "baseline":  "Baseline\n(gốc)",
    "jpeg_q70":  "JPEG q=70",
    "jpeg_q50":  "JPEG q=50",
    "jpeg_q30":  "JPEG q=30",
    "resize_50": "Resize 50%",
    "resize_25": "Resize 25%",
    "blur_s1":   "Blur σ=1.0",
    "blur_s2":   "Blur σ=2.0",
}

fig, axes = plt.subplots(2, 8, figsize=(22, 6))
for row_idx, (path, row_label) in enumerate([(_real_path, "Real"), (_fake_path, "Fake")]):
    try:
        img = Image.open(path).convert("RGB")
    except Exception:
        img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    for col_idx, name in enumerate(_perturb_order):
        perturbed = PERTURBATIONS[name](img.copy())
        ax = axes[row_idx][col_idx]
        ax.imshow(perturbed)
        ax.axis("off")
        if row_idx == 0:
            ax.set_title(_labels[name], fontsize=9,
                         fontweight="bold" if name == "baseline" else "normal")
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=11, fontweight="bold", rotation=0,
                          labelpad=35, va="center")

fig.suptitle("Ảnh mẫu trước/sau từng perturbation (hàng trên: Real, hàng dưới: Fake)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print(f"Real: {_real_path}")
print(f"Fake: {_fake_path}")

In [ ]:
# Cell 3 — RobustnessDataset
class RobustnessDataset(Dataset):
    def __init__(self, df, perturb_fn, transform):
        self.df         = df.reset_index(drop=True)
        self.perturb_fn = perturb_fn
        self.transform  = transform
        self.label_map  = {"Real": 0, "Fake": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            img = self.perturb_fn(img)
        except Exception:
            img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        img_np = np.array(img)
        img_t  = self.transform(image=img_np)["image"]
        label  = torch.tensor(self.label_map[row["label"]], dtype=torch.float32)
        return img_t, label


In [ ]:
# Cell 4 — Load model & transform
_src = "src" if os.path.isdir("src") else "../src"
if os.path.abspath(_src) not in sys.path:
    sys.path.insert(0, os.path.abspath(_src))

from model import FaceDetectionModel


def load_model(checkpoint_path, device):
    model = FaceDetectionModel(
        backbone_name="efficientnet_b0",
        pretrained=False,
        hidden_dim=256,
        dropout=0.5,
    ).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(
        f"Loaded checkpoint — epoch {ckpt['epoch']}, "
        f"val_loss {ckpt['val_loss']:.4f}, val_acc {ckpt['val_acc']:.4f}"
    )
    model.eval()
    return model


model   = load_model(CHECKPOINT_PATH, DEVICE)
test_df = pd.read_csv(TEST_CSV)
print(f"Test set: {len(test_df)} mẫu")

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

In [ ]:
# Cell 5 — Đánh giá qua từng perturbation
robustness_results = {}

for name, perturb_fn in PERTURBATIONS.items():
    dataset = RobustnessDataset(test_df, perturb_fn, val_transform)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    all_preds, all_probs, all_labels = [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f"[{name}]", leave=False):
            logits = model(imgs.to(DEVICE)).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= 0.5).astype(int)
            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())

    y    = np.array(all_labels)
    p    = np.array(all_preds)
    prob = np.array(all_probs)
    robustness_results[name] = {
        "accuracy":  float(accuracy_score(y, p)),
        "precision": float(precision_score(y, p, zero_division=0)),
        "recall":    float(recall_score(y, p, zero_division=0)),
        "f1":        float(f1_score(y, p, zero_division=0)),
        "auc":       float(roc_auc_score(y, prob)),
    }
    m = robustness_results[name]
    print(
        f"[{name:9s}]  Accuracy={m['accuracy']:.4f}  Precision={m['precision']:.4f}  "
        f"Recall={m['recall']:.4f}  F1={m['f1']:.4f}  AUC={m['auc']:.4f}"
    )

## Phân tích kết quả Robustness

| Perturbation | Accuracy | AUC-ROC | Drop | Đánh giá |
|---|---|---|---|---|
| Baseline (gốc) | **0.9733** | **0.9970** | 0.000 | — |
| JPEG q=70 | 0.8233 | 0.9597 | −0.150 | 🔴 Nhạy cảm đáng kể |
| JPEG q=50 | 0.7977 | 0.9330 | −0.176 | 🔴 Nhạy cảm đáng kể |
| JPEG q=30 | 0.7860 | 0.9030 | −0.187 | 🔴 Nhạy cảm đáng kể |
| Resize 50% | 0.8539 | 0.9358 | −0.119 | 🔴 Nhạy cảm đáng kể |
| Resize 25% | 0.7287 | 0.8040 | **−0.245** | 🔴 Nhạy cảm nhất |
| Blur σ=1.0 | 0.9045 | 0.9776 | −0.069 | 🔴 Nhạy cảm đáng kể |
| Blur σ=2.0 | 0.7365 | 0.8211 | −0.237 | 🔴 Nhạy cảm đáng kể |

### Nhận xét theo từng loại perturbation

**JPEG Compression (jpeg_q70/q50/q30 — Drop: −15.0 / −17.6 / −18.7 pp):**
- Sụt giảm tỷ lệ thuận với độ nén: chất lượng càng thấp thì accuracy càng giảm.
- **AUC vẫn cao** (0.960/0.933/0.903) dù accuracy giảm mạnh — mô hình vẫn duy trì khả năng *xếp hạng* tốt, nhưng ngưỡng 0.5 không còn tối ưu sau khi artifact tần số cao bị phá hủy bởi nén.
- **Cơ chế:** JPEG xóa các chi tiết tần số cao — đây chính là loại artifact đặc trưng của GAN/deepfake mà mô hình phụ thuộc để phân loại.

**Resize Down-Up (resize_50/resize_25 — Drop: −11.9 / −24.5 pp):**
- Resize 25% là perturbation **gây hại nhất toàn bộ thực nghiệm** (Drop −24.5 pp, AUC giảm xuống 0.804).
- Khi thu nhỏ 75%, phần lớn micro-texture và artifact bị mất hoàn toàn qua bilinear interpolation. Ở mức này ngay cả khả năng ranking cũng bị ảnh hưởng nghiêm trọng.

**Gaussian Blur (blur_s1/s2 — Drop: −6.9 / −23.7 pp):**
- Khoảng cách giữa σ=1.0 và σ=2.0 rất lớn (~16.8 pp) — cho thấy mô hình cực kỳ nhạy với mức độ blur.
- Blur nhẹ (σ=1.0): vẫn đạt AUC=0.9776, nhiều đặc trưng hữu ích còn lại.
- Blur mạnh (σ=2.0): mất gần toàn bộ fine-grained detail, accuracy giảm xuống 0.7365.

### Kết luận chính
**Không một perturbation nào đạt ngưỡng chấp nhận được (Drop ≤ 5 pp).** Mô hình học đặc trưng tần số cao chất lượng cao nhưng chưa được trang bị augmentation để chịu đựng nhiễu thực tế (JPEG, blur, downscale).

In [ ]:
# Cell 6 — Lưu kết quả & vẽ biểu đồ
with open(f"{OUTPUT_DIR}/robustness_metrics.json", "w", encoding="utf-8") as f:
    json.dump(robustness_results, f, indent=2)
print("Đã lưu:", f"{OUTPUT_DIR}/robustness_metrics.json")

ORDER    = ["baseline", "jpeg_q70", "jpeg_q50", "jpeg_q30",
            "resize_50", "resize_25", "blur_s1", "blur_s2"]
names    = [n for n in ORDER if n in robustness_results]
acc_vals = [robustness_results[n]["accuracy"] for n in names]
auc_vals = [robustness_results[n]["auc"]      for n in names]
baseline_acc = robustness_results["baseline"]["accuracy"]
baseline_auc = robustness_results["baseline"]["auc"]

# Line chart Accuracy
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(names, acc_vals, marker="o", linewidth=2, color="#4C9BE8", label="Accuracy")
ax.axhline(baseline_acc, linestyle="--", color="#999", linewidth=1,
           label=f"Baseline ({baseline_acc:.3f})")
for n, v in zip(names, acc_vals):
    ax.annotate(f"{v:.3f}", (n, v),
                textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=8)
ax.set_ylim(max(0, min(acc_vals) - 0.05), 1.02)
ax.set_xlabel("Perturbation")
ax.set_ylabel("Accuracy")
ax.set_title("Robustness Test — Accuracy theo Perturbation")
ax.legend()
ax.tick_params(axis="x", rotation=20)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/robustness_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved robustness_accuracy.png")

# Line chart AUC
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(names, auc_vals, marker="s", linewidth=2, color="#E87C4C", label="AUC-ROC")
ax.axhline(baseline_auc, linestyle="--", color="#999", linewidth=1,
           label=f"Baseline ({baseline_auc:.3f})")
for n, v in zip(names, auc_vals):
    ax.annotate(f"{v:.3f}", (n, v),
                textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=8)
ax.set_ylim(max(0, min(auc_vals) - 0.05), 1.02)
ax.set_xlabel("Perturbation")
ax.set_ylabel("AUC-ROC")
ax.set_title("Robustness Test — AUC-ROC theo Perturbation")
ax.legend()
ax.tick_params(axis="x", rotation=20)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/robustness_auc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved robustness_auc.png")

# Drop chart
acc_drops = [baseline_acc - v for v in acc_vals]
colors    = ["#4CE87C" if d <= 0.02 else "#E8C44C" if d <= 0.05 else "#E84C4C"
             for d in acc_drops]
fig, ax   = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, acc_drops, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(0.05, color="#E8C44C", linestyle=":", linewidth=1, alpha=0.8, label="Ngưỡng 5 pp")
ax.axhline(0.02, color="#4CE87C", linestyle=":", linewidth=1, alpha=0.8, label="Ngưỡng 2 pp")
ax.set_xlabel("Perturbation")
ax.set_ylabel("Accuracy Drop (so với baseline)")
ax.set_title("Robustness Test — Mức sụt giảm Accuracy\n(xanh ≤2 pp | vàng ≤5 pp | đỏ >5 pp)")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=9)
for bar, v in zip(bars, acc_drops):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/robustness_drop.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved robustness_drop.png")

print(f"\nTất cả kết quả đã lưu vào: {OUTPUT_DIR}")
print("\n=== TÓM TẮT ===")
for name in names:
    m = robustness_results[name]
    drop = baseline_acc - m["accuracy"]
    print(f"  {name:12s}  Acc={m['accuracy']:.4f}  AUC={m['auc']:.4f}  Drop={drop:+.4f}")

## Kết luận và hướng cải thiện

### Tóm tắt

- **Điểm mạnh:** Trên ảnh sạch (baseline), mô hình đạt Accuracy=0.9733, AUC=0.9970 — xuất sắc.
- **Điểm yếu cốt lõi:** Mô hình nhạy cảm với **mọi** dạng suy giảm chất lượng. Không một perturbation nào trong ngưỡng chấp nhận được (≤5 pp). Trong môi trường thực tế — nơi hầu hết ảnh đều qua ít nhất một lần nén JPEG — hiệu suất thực sẽ thấp hơn đáng kể so với con số 97.33%.
- **Worst case:** Resize 25% (Drop −24.5 pp) và Blur σ=2.0 (Drop −23.7 pp) — mức mà ảnh chụp lại từ màn hình hoặc chia sẻ qua ứng dụng nhắn tin có thể gặp phải.

### Hướng cải thiện (augmentation trong train)

| Vấn đề | Augmentation đề xuất |
|--------|---------------------|
| JPEG sensitivity | `albumentations.ImageCompression(quality_lower=40, quality_upper=95)` |
| Resize sensitivity | `albumentations.Downscale(scale_min=0.25, scale_max=0.75)` |
| Blur sensitivity | `albumentations.GaussianBlur(blur_limit=(3, 7))` |
| Kết hợp | Áp dụng ngẫu nhiên 1 trong 3 với xác suất 50% mỗi batch |

Thêm các augmentation này không thay đổi kiến trúc, chỉ cần retrain từ Stage 2 hoặc Stage 3 với transform mới.